# Lab W5 (WAJIB): RNN vs LSTM pada Sequence

## Pre-flight Checklist

> [!IMPORTANT]
> Lab ini **wajib** untuk W5. Memenuhi Breadth Check keluarga RNN/LSTM (lihat Kontrak Belajar §6 di Pendahuluan). Konsep yang ditandai (§) merujuk ke `05_W5_Sequences_RNN_LSTM.md`.

**Yang Anda butuhkan sebelum mulai:**
- Bab W5 sudah dibaca, terutama §1.5 (BPTT primer + tabel vanishing numerik), §2.2 (RNN vanilla annotated), §2.3 (LSTM gates + Hadamard primer + forget gate intuisi + hidden vs cell state).
- Familiar dengan tuple shape `(B, T, F)` untuk batch sequence (lihat §0.5.1 Pendahuluan).
- Familiar dengan gradient clipping (`torch.nn.utils.clip_grad_norm_`) - default di lab ini, bukan opsional.

**Yang akan Anda hasilkan di akhir lab:**
- Plot gradient norm log-scale: vanilla RNN turun eksponensial, LSTM relatif flat (bukti vanishing gradient §1.5.2).
- LSTM terlatih untuk sine + noise forecasting; val MSE stabil < 0.02.
- Visualisasi prediksi vs ground truth pada 6 sampel validasi.
- Refleksi yang menghubungkan eksperimen ke teori vanishing gradient.

**Resource:**
- **Hardware:** CPU cukup (sequence pendek, model kecil). GPU mempercepat tetapi tidak wajib.
- **Estimasi waktu kerja:** 3-4 jam termasuk membaca, eksekusi, dan refleksi.

**Pendamping:** Bab W5 di `05_W5_Sequences_RNN_LSTM.md`.

**Tujuan pedagogis:** mendemonstrasikan secara visual perbedaan gradient flow antara vanilla RNN dan LSTM, lalu melatih LSTM untuk one-step-ahead prediction pada sinyal sine + noise sintetis. Dataset sintetis dipilih karena: (a) tidak perlu download, (b) struktur jelas, model sehat akan konvergen, (c) noise dapat divariasikan untuk diagnostik.

## 1. Setup dan smoke test

In [ ]:
# ── Setup ────────────────────────────────────────────────────────────────────
# Di Google Colab: clone repo terlebih dahulu, lalu cd ke template_repo/
#   git clone <url-repo> && cd template_repo
# Di environment lokal: jalankan dari folder notebooks/
import sys, os
_root = os.path.abspath('..')
if _root not in sys.path:
    sys.path.insert(0, _root)
print("Root:", _root)

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

from src.models import SimpleLSTM

torch.manual_seed(42)
np.random.seed(42)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)

## 2. Dataset sintetis: sine + noise

Kita bangun dataset forecasting sederhana - diberi 50 langkah sebelumnya, prediksi langkah ke-51. Dataset sintetis dipilih karena: (a) tidak perlu download, (b) pola struktur pasti sehingga kita tahu model "yang jujur bekerja" akan konvergen, (c) bisa divariasikan tingkat noise-nya untuk diagnostik.

In [ ]:
def make_sine_dataset(n_samples=4000, seq_len=50, noise=0.05, seed=42):
    rng = np.random.default_rng(seed)
    starts = rng.uniform(0, 2 * np.pi, size=n_samples)
    freqs = rng.uniform(0.5, 2.0, size=n_samples)
    t = np.linspace(0, 4 * np.pi, seq_len + 1)
    data = np.sin(starts[:, None] + freqs[:, None] * t)  # (n, seq_len+1)
    data += rng.normal(0, noise, data.shape)
    X = data[:, :-1][:, :, None].astype(np.float32)      # (n, seq_len, 1)
    y = data[:, -1:].astype(np.float32)                  # (n, 1)
    return X, y

X_tr, y_tr = make_sine_dataset(4000, 50, noise=0.05, seed=42)
X_va, y_va = make_sine_dataset(500, 50, noise=0.05, seed=1337)

print('X train:', X_tr.shape, 'y train:', y_tr.shape)
plt.plot(X_tr[0, :, 0]); plt.scatter([50], y_tr[0], color='red', label='target'); plt.legend(); plt.title('satu sampel')
plt.show()

## 3. Vanilla RNN vs LSTM: gradient flow

Sebelum training, kita ilustrasikan *vanishing gradient* dengan backpropagate gradient dari loss di timestep terakhir ke setiap timestep input, lalu plot norm gradient per-timestep. Untuk RNN dalam, gradient di timestep awal biasanya nyaris nol - itulah kenapa LSTM didesain.

In [ ]:
def gradient_norm_per_timestep(cell_type='RNN', seq_len=50, hidden=32):
    torch.manual_seed(0)
    if cell_type == 'RNN':
        cell = nn.RNN(1, hidden, batch_first=True)
    else:
        cell = nn.LSTM(1, hidden, batch_first=True)
    x = torch.randn(1, seq_len, 1, requires_grad=True)
    out, _ = cell(x)
    loss = out[:, -1, :].sum()
    loss.backward()
    g = x.grad[0, :, 0].abs().numpy()
    return g

g_rnn  = gradient_norm_per_timestep('RNN')
g_lstm = gradient_norm_per_timestep('LSTM')
plt.plot(g_rnn,  label='vanilla RNN', marker='.')
plt.plot(g_lstm, label='LSTM',        marker='.')
plt.yscale('log'); plt.xlabel('timestep (0 paling lama, 49 terbaru)')
plt.ylabel('|gradient| di input'); plt.title('Gradient flow lewat sequence'); plt.legend()
plt.show()
print('Gradient awal RNN:  ', g_rnn[0], '| akhir:', g_rnn[-1])
print('Gradient awal LSTM: ', g_lstm[0], '| akhir:', g_lstm[-1])

**Apa yang Anda perhatikan di plot gradient flow?**

Ekspektasi (sesuai §1.5.2 W5 dan tabel `w_h^T`):

- **Vanilla RNN.** Gradient di timestep awal (indeks 0) harus **jauh lebih kecil** dari gradient di timestep akhir (indeks 49) - urutan magnitudo `1e-3` atau lebih kecil. Pola turun eksponensial (mengikuti garis lurus di y-log scale). Inilah vanishing gradient.
- **LSTM.** Gradient relatif flat sepanjang sequence; gradient di timestep awal sebanding dengan timestep akhir (urutan magnitudo sama). Cell state highway memutus rantai perkalian matriks.

**Pertanyaan diagnostik:**

- Berapa rasio `g_rnn[0] / g_rnn[-1]`? Untuk seq_len=50, ekspektasi ~ 1e-3 sampai 1e-5.
- Berapa rasio `g_lstm[0] / g_lstm[-1]`? Ekspektasi ~ 0.1 sampai 1.0 (dalam orde sama).
- Coba ulangi dengan `seq_len=200`. Bagaimana rasio berubah untuk RNN vs LSTM?

---

## 4. Training loop

Pakai `SimpleLSTM` dari `src/models.py` agar reproducibility (seed, config) konsisten dengan praktik Bab 03.

In [ ]:
from torch.utils.data import DataLoader, TensorDataset

model = SimpleLSTM(input_size=1, hidden_size=64, num_layers=1, num_classes=1, readout='last').to(device)
opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-5)
crit = nn.MSELoss()

ds_tr = TensorDataset(torch.tensor(X_tr), torch.tensor(y_tr))
dl_tr = DataLoader(ds_tr, batch_size=64, shuffle=True)
X_va_t = torch.tensor(X_va).to(device); y_va_t = torch.tensor(y_va).to(device)

hist_tr, hist_va = [], []
for epoch in range(20):
    model.train(); running = 0.0
    for Xb, yb in dl_tr:
        Xb, yb = Xb.to(device), yb.to(device)
        opt.zero_grad()
        loss = crit(model(Xb), yb)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)  # selalu clip pada sequence model
        opt.step()
        running += loss.item() * len(Xb)
    model.eval()
    with torch.no_grad():
        va_loss = crit(model(X_va_t), y_va_t).item()
    hist_tr.append(running / len(ds_tr)); hist_va.append(va_loss)
    print(f'epoch {epoch+1:2d}  tr={hist_tr[-1]:.4f}  va={va_loss:.4f}')

In [ ]:
plt.plot(hist_tr, label='train'); plt.plot(hist_va, label='val')
plt.xlabel('epoch'); plt.ylabel('MSE'); plt.legend(); plt.title('LSTM sine forecasting')
plt.show()

## 5. Inspeksi prediksi

Visualisasi: prediksi vs ground truth pada 6 sampel validasi.

## 6. Refleksi

### 6.1 Pertanyaan Tertarget

1. **Vanishing gradient quantified.** Pada cell §3, hitung rasio `g_rnn[0] / g_rnn[-1]` dan `g_lstm[0] / g_lstm[-1]`. Apakah hasil Anda konsisten dengan tabel `w_h^T` di §1.5.2 W5? Pilih jawaban paling akurat:
   - (a) Rasio RNN < 1e-3, rasio LSTM ~ orde-1. Konsisten dengan vanishing gradient.
   - (b) Rasio RNN dan LSTM keduanya ~ orde-1. Vanishing gradient tidak terlihat di seq_len=50 (perlu seq_len lebih panjang).
   - (c) Rasio RNN ~ orde-1, rasio LSTM < 1e-3. Hasil tidak sesuai prediksi - ada bug di setup.
   - (d) Hasil bervariasi antar run (`torch.manual_seed(0)` tidak cukup deterministik).

2. **Gradient clipping eksperimen.** Komen-out `torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)` di training loop, latih ulang. Apakah loss tetap stabil, atau muncul lonjakan/NaN? Pada titik mana (epoch berapa) gejala muncul? Hubungkan dengan §1.5.2 W5 (rezim `|w_h| > 1` exploding).

3. **MLP baseline.** Ganti `SimpleLSTM` dengan `SimpleMLP(input_dim=50, hidden_sizes=(64,), num_classes=1)` dan flatten input `Xb.flatten(start_dim=1)`. Latih 20 epoch. Apakah val MSE sebanding, lebih baik, atau jauh lebih buruk? Hubungkan dengan §2.2 W5 (RNN family vs alternatives) dan strategi representasi W3 §2.4.

### 6.2 Self-Check Quick

- [ ] Plot gradient flow menunjukkan vanishing pada RNN, flat pada LSTM (rasio ≥ 100× berbeda).
- [ ] LSTM training konvergen; val MSE < 0.02 di epoch 20.
- [ ] Visualisasi prediksi 6 sampel terlihat masuk akal (prediksi dekat ground truth merah).
- [ ] Saya bisa menjelaskan dengan kata-kata sendiri kenapa cell state di LSTM memutus rantai vanishing gradient.
- [ ] Gradient clipping aktif sebelum `optimizer.step()`.

## 6. Refleksi

1. Pada cell 3, gradient timestep awal (indeks 0) seberapa besar untuk RNN vs LSTM? Apakah hasil Anda konsisten dengan prediksi *vanishing gradient* di Section 2.0b?
2. Coba matikan `clip_grad_norm_` pada training. Apakah loss masih stabil? Di titik mana (kalau terjadi) Anda melihat sinyal *exploding gradient*?
3. Gantikan `SimpleLSTM` dengan `SimpleMLP(input_dim=50, hidden_sizes=(64,), num_classes=1)` (dengan `x.flatten(start_dim=1)`). Apakah MLP bisa mencapai val loss yang sama? Mengapa atau mengapa tidak? Hubungkan jawaban Anda dengan *efisiensi statistik* yang dibahas di akhir Section 2.0b.